# CAMP Notebook for the SurgBioMech Lab

### Preprocessing

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from thoracic_prepro import *

In [ ]:
spec_batch = None
log = thoracic_preprocessing_batch(
    cohorts=["JP"],
    base_root=r"Z:\aorta\thoracic",
    iso_spacing_mm=3.0,
    buffer_mm=5.0,
    target_faces=50_000,
    individ_scans=spec_batch,
    include_centerline_in_log=False,
    continue_on_error=True,
)
log

In [ ]:
#log[log['euler_number'] == 1.0]['scan_id'].unique()

In [ ]:
import numpy as np
import pandas as pd
import trimesh
import plotly.graph_objects as go

scan_to_view = "KP8_4"  # change as needed

if "log" not in globals() or not isinstance(log, pd.DataFrame) or log.empty:
    raise ValueError("Run Cell 4 first so 'log' is populated.")

row = log.loc[log["scan_id"] == scan_to_view]
if row.empty:
    raise ValueError(f"scan_id '{scan_to_view}' not found in log. Available: {log['scan_id'].tolist()}")
row = row.iloc[0]

if row.get("status", "") != "ok":
    raise ValueError(f"Scan '{scan_to_view}' did not finish with status='ok': {row.get('status')} | {row.get('error')}")

mesh_path = row["mat_file"]
centerline_pts = row.get("centerline_points_xyz", None)

if centerline_pts is None:
    raise ValueError(
        "No centerline data in log. Rerun Cell 4 with include_centerline_in_log=True."
    )

centerline_pts = np.asarray(centerline_pts, dtype=float)
if centerline_pts.ndim != 2 or centerline_pts.shape[1] != 3:
    raise ValueError("centerline_points_xyz has unexpected shape; expected (N, 3).")

mesh, _ = load_surface_from_mat(mesh_path, scan_prefix=None)
v = mesh.vertices
f = mesh.faces

mesh_trace = go.Mesh3d(
    x=v[:, 0], y=v[:, 1], z=v[:, 2],
    i=f[:, 0], j=f[:, 1], k=f[:, 2],
    color="lightgray",
    opacity=0.35,
    name="50K mesh",
    showscale=False,
)

cl_trace = go.Scatter3d(
    x=centerline_pts[:, 0],
    y=centerline_pts[:, 1],
    z=centerline_pts[:, 2],
    mode="lines+markers",
    line=dict(color="red", width=8),
    marker=dict(size=2, color="red"),
    name="centerline",
)

start_trace = go.Scatter3d(
    x=[centerline_pts[0, 0]], y=[centerline_pts[0, 1]], z=[centerline_pts[0, 2]],
    mode="markers",
    marker=dict(size=7, color="blue", symbol="circle"),
    name="start",
)

end_trace = go.Scatter3d(
    x=[centerline_pts[-1, 0]], y=[centerline_pts[-1, 1]], z=[centerline_pts[-1, 2]],
    mode="markers",
    marker=dict(size=7, color="green", symbol="diamond"),
    name="end",
)

fig = go.Figure(data=[mesh_trace, cl_trace, start_trace, end_trace])
fig.update_layout(
    title=f"Centerline QC: {scan_to_view}",
    scene=dict(
        aspectmode="data",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
    ),
    width=900,
    height=700,
)
fig.show()

### Curvature Adaptive Monge Patching (CAMP)

An adaptive patch-based remeshing workflow that uses intrinsic surface patch quantities to partition the surface into local geometric-constrained domains:

- Uses local geodesic neighborhoods and Monge 2nd-order polynomial fits.
- Builds adaptive geodesic patches with curvature-weighted target patch sizes.
- Computes both:
  - average discrete Gaussian curvature per patch (`avg_kg`)
  - integrated Gaussian curvature per patch (`integrated_K`)
- Run terminates only when **every patch in the queue either passes both quality thresholds (`R2 ≥ R2_thresh` AND `η ≤ eta_thresh`) or is too small to split further (`< 2 × min_patch_size` vertices)**. The final patch count is a function of the data, not a user-set target.

In [ ]:
from campsrc import *

In [ ]:
scan_id = "KK29_3"
MAT_PATH = r"Z:\aorta\thoracic\KK\KK29\KK29_3\mesh\KK29_3_o50K.mat"
COLOR_METRIC_CONV = "integrated_K"
COLOR_BOUNDS_CONV = (-0.5, 0.5)

out = camp(
    mat_path=MAT_PATH,
    scan_prefix=None,
    color_metric=COLOR_METRIC_CONV,
    color_bounds=COLOR_BOUNDS_CONV,
    R2_thresh=0.9,
    eta_thresh=0.075,
    min_patch_size=18,
    seed_patches=100,
    trim_frac=0.001,
    color_scale="Inferno",
)
out["figure"].show()

# Param sweeps

In [ ]:
csweep = camp_param_sweep(
    cohorts=("AV", "CT"),
    base_root=r"Z:\aorta\thoracic",
    mat_suffix="_o50",
    r2_thresh=0.9,
    eta_thresh=0.075,
    min_patch_size=18,
    seed_patches=100,
    trim_fracs=(0.0, 0.001, 0.005, 0.01, 0.05),
    n_jobs=6,
)

In [ ]:
# Sanity checks for exact sweep request
expected_cols = [
    "r2_thresh", "eta_thresh", "min_patch_size", "seed_patches",
    "trim_frac", "scan_id", "surface_area_mm2", "area_total", "k_fluct",
    "n_patches_trimmed", "area_frac_trimmed", "n_patches", "euler_number",
    "is_straw", "pct_failed_quality", "pct_at_min_size",
]

csweep = csweep[expected_cols].copy()

print("rows:", len(csweep))
print("columns:", csweep.columns.tolist())
print("unique seeds:", sorted(csweep["seed_patches"].dropna().unique().tolist()))
print("unique trim_fracs:", sorted(csweep["trim_frac"].dropna().unique().tolist()))
print("k_fluct missing:", int(csweep["k_fluct"].isna().sum()))

In [ ]:
csweep.keys()

In [ ]:
# Analysis table from cleaned sweep output
csweep2 = csweep.copy()

In [ ]:
csweep2

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# SWEEP_CSV = r"Z:\aorta\thoracic\_sweep\sweep_master2.csv"

# df = pd.read_csv(SWEEP_CSV)
df = csweep3.copy()
df["cohort"] = df["scan_id"].astype(str).str.extract(r"^([A-Za-z]+)")[0]

PARAM_COLS = ["r2_thresh", "eta_thresh", "min_patch_size", "seed_patches", "trim_frac"]
opts = {p: sorted(df[p].dropna().unique().tolist()) for p in PARAM_COLS}

# numeric, non-parameter columns are the plottable features
feature_cols = [
    c for c in df.columns
    if c not in PARAM_COLS + ["scan_id", "cohort"] and pd.api.types.is_numeric_dtype(df[c])
]

def _default(col, fallback):
    return col if col in feature_cols else fallback

x_dd = widgets.Dropdown(options=feature_cols, value=_default("n_patches", feature_cols[0]), description="x:")
y_dd = widgets.Dropdown(options=feature_cols, value=_default("k_fluct", feature_cols[0]), description="y:")
lock = widgets.Checkbox(value=True, description="lock axes (global range)")
straws_only = widgets.Checkbox(value=("is_straw" in df.columns), description="straws only")

sliders = {
    p: widgets.SelectionSlider(options=opts[p], value=opts[p][len(opts[p]) // 2],
                               description=p, continuous_update=False,
                               style={"description_width": "120px"},
                               layout=widgets.Layout(width="480px"))
    for p in PARAM_COLS
}

cohorts = sorted(df["cohort"].dropna().unique().tolist())
#palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]
# palette = ["#000000", "#14955D", "#ff0000", "#9467bd", "#ff7f0e", "#8c564b", "#1f77b4", "#d62728", "#2ca02c"]
# names = ['GORE 0801', 'UC TBAD', 'Medtronic', 'Normal Controls', 'French MRA', 'AV', 'CT']

fig = go.FigureWidget()
for i, c in enumerate(cohorts):
    if c == "MF":
        fig.add_scatter(x=[], y=[], mode="markers", name="French MRA",
                        marker=dict(size=10, opacity=1, color="#8400ff"))
    if c == "KY":
        fig.add_scatter(x=[], y=[], mode="markers", name="Normal Controls",
                        marker=dict(size=5, opacity=0.5, color="#1F763A"))
    if c == "AV":
        fig.add_scatter(x=[], y=[], mode="markers", name="GORE 0802",
                        marker=dict(size=5, opacity=1, color="#49D397"))
    if c == "CT":
        fig.add_scatter(x=[], y=[], mode="markers", name="Cook",
                        marker=dict(size=5, opacity=1, color="#ff0000"))
    if c == "KK":
        fig.add_scatter(x=[], y=[], mode="markers", name="UC TBAD",
                        marker=dict(size=5, opacity=1, color="#ff7f0e"))
    if c == "DJ":
        fig.add_scatter(x=[], y=[], mode="markers", name="GORE 0801",
                        marker=dict(size=5, opacity=1, color="#0d00ff"))
    if c == "KP":
        fig.add_scatter(x=[], y=[], mode="markers", name="Medtronic",
                        marker=dict(size=5, opacity=1, color="#acb800"))

fig.update_layout(width=820, height=600, template="plotly_white",
                  legend=dict(title="cohort"), margin=dict(l=60, r=20, t=60, b=50))

def update(*_):
    sel = df
    for p in PARAM_COLS:
        sel = sel[sel[p] == sliders[p].value]
    if straws_only.value and "is_straw" in df.columns:
        sel = sel[sel["is_straw"].astype(bool)]
    xcol, ycol = x_dd.value, y_dd.value
    with fig.batch_update():
        for i, c in enumerate(cohorts):
            d = sel[sel["cohort"] == c]
            fig.data[i].x = d[xcol]
            fig.data[i].y = d[ycol]
            fig.data[i].text = d["scan_id"]
            fig.data[i].hovertemplate = f"%{{text}}<br>{xcol}=%{{x:.4g}}<br>{ycol}=%{{y:.4g}}<extra>{c}</extra>"
        fig.layout.xaxis.title = xcol
        fig.layout.yaxis.title = ycol
        if lock.value:
            fig.layout.xaxis.range = [df[xcol].min(), df[xcol].max()]
            fig.layout.yaxis.range = [df[ycol].min(), df[ycol].max()]
        else:
            fig.layout.xaxis.autorange = True
            fig.layout.yaxis.autorange = True
        params = "  ".join(f"{p}={sliders[p].value}" for p in PARAM_COLS)
        fig.layout.title = f"{params}   (n={len(sel)})"

for w in [x_dd, y_dd, lock, straws_only] + list(sliders.values()):
    w.observe(update, names="value")
update()

display(widgets.VBox([
    widgets.HBox([x_dd, y_dd]),
    widgets.HBox([lock, straws_only]),
    *[sliders[p] for p in PARAM_COLS],
    fig,
]))

In [ ]:
csweep2.to_csv('trauma_cook_trials_camp71526.csv')

In [ ]:
one = pd.read_csv(r"Z:\aorta\python_code\camp\trauma_cook_trials_camp71526.csv")
two = pd.read_csv(r"Z:\aorta\python_code\camp\UC_plus_acute_trials_camp71426.csv")
csweep3 = pd.concat([one, two], ignore_index=True)

## Comparing Milou's method and UC's manual segmentations

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_scan_pairs(
    df,
    x,
    y,
    group="scan_id",
    cohort_col="cohort",
    ref_cohort="milou",
    cmp_cohort="uc",
    colors=None,
    point_size=9,
    ky_size=4,
    line_color="rgba(150,150,150,0.55)",
    width=1100,
    height=500,
):
    """
    Parameters
    ----------
    df : pandas.DataFrame
    x, y : str
        Column names for the left-panel axes. `y` is also the value compared
        in the parity panel.
    group : str
        Column whose shared values are joined by a line (default 'scan_id').
        Also the key used to pair cohorts.
    cohort_col : str
        Column holding the cohort label (default 'cohort').
    ref_cohort, cmp_cohort : str
        Cohort labels placed on the parity x- and y-axes respectively
        (default 'milou' on x, 'uc' on y). Matched case-insensitively. A
        milou/uc point appears on the left panel only if its scan_id has a
        value in both cohorts.
    colors : dict or None
        Overrides for point colors. Keys: ref_cohort, cmp_cohort, "KY", "other".
    point_size, ky_size : int
        Marker sizes for normal points and for KY-tagged points.
    line_color : str
        Color of the scan_id connecting lines.
 
    Returns
    -------
    plotly.graph_objects.Figure
    """
    colors = colors or {
        ref_cohort: "red",
        cmp_cohort: "blue",
        "KY": "green",
        "other": "lightgray",
    }
 
    fig = make_subplots(rows=1, cols=2)
 
    # Cohort labels (case-insensitive) and the scan_ids paired across cohorts.
    coh = df[cohort_col].astype(str).str.strip().str.lower()
    ref = df[coh == ref_cohort.lower()].groupby(df[group])[y].mean()
    cmp = df[coh == cmp_cohort.lower()].groupby(df[group])[y].mean()
    pair = pd.concat([ref.rename("ref"), cmp.rename("cmp")], axis=1).dropna()
    paired_ids = set(pair.index)
 
    # --- Left: connecting lines for paired scan_ids only ---
    xs, ys = [], []
    for _, sub in df[df[group].isin(paired_ids)].groupby(group, sort=True):
        sub = sub.sort_values(x)
        xs.extend(sub[x].tolist() + [None])
        ys.extend(sub[y].tolist() + [None])
    fig.add_trace(
        go.Scatter(
            x=xs, y=ys, mode="lines",
            line=dict(color=line_color, width=1),
            hoverinfo="skip", showlegend=False,
        ),
        row=1, col=1,
    )
 
    # --- Left: points. milou/uc only when paired; KY overrides cohort color ---
    is_ky = df[group].astype(str).str.contains("KY", case=True, regex=False)
    cat = pd.Series("other", index=df.index)
    cat[coh == ref_cohort.lower()] = ref_cohort
    cat[coh == cmp_cohort.lower()] = cmp_cohort
    cat[is_ky] = "KY"
 
    style = {
        ref_cohort: (colors[ref_cohort], point_size, ref_cohort),
        cmp_cohort: (colors[cmp_cohort], point_size, cmp_cohort),
        "KY": (colors["KY"], ky_size, "KY (scan_id)"),
        "other": (colors["other"], point_size, "other"),
    }
    paired_mask = df[group].isin(paired_ids)
    for c, (color, size, name) in style.items():
        m = cat == c
        if c in (ref_cohort, cmp_cohort):      # red/blue must be paired
            m = m & paired_mask
        if not m.any():
            continue
        fig.add_trace(
            go.Scatter(
                x=df.loc[m, x], y=df.loc[m, y], mode="markers",
                marker=dict(color=color, size=size,
                            line=dict(width=0.5, color="white")),
                name=name,
                customdata=df.loc[m, [group, cohort_col]].to_numpy(),
                hovertemplate=(
                    f"{group}=%{{customdata[0]}}<br>{cohort_col}=%{{customdata[1]}}"
                    f"<br>{x}=%{{x}}<br>{y}=%{{y}}<extra></extra>"
                ),
            ),
            row=1, col=1,
        )
 
    # --- Right: parity of y between cohorts, paired on `group` ---
    xv, yv = pair["ref"].to_numpy(), pair["cmp"].to_numpy()
    if len(xv):
        fig.add_trace(
            go.Scatter(
                x=xv, y=yv, mode="markers",
                marker=dict(color="black", size=7),
                text=pair.index.astype(str),
                hovertemplate=(
                    f"{group}=%{{text}}<br>{ref_cohort}=%{{x}}"
                    f"<br>{cmp_cohort}=%{{y}}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=1, col=2,
        )
 
        lo = float(min(xv.min(), yv.min()))
        hi = float(max(xv.max(), yv.max()))
        pad = 0.05 * (hi - lo or 1.0)
        lo, hi = lo - pad, hi + pad
 
        fig.add_trace(
            go.Scatter(
                x=[lo, hi], y=[lo, hi], mode="lines",
                line=dict(color="gray", dash="dash", width=1),
                showlegend=False, hoverinfo="skip",
            ),
            row=1, col=2,
        )
        fig.update_xaxes(range=[lo, hi], row=1, col=2)
        fig.update_yaxes(range=[lo, hi], scaleanchor="x2", scaleratio=1,
                         row=1, col=2)
 
        if len(xv) >= 2:
            ss_res = float(np.sum((xv - yv) ** 2))
            ss_tot = float(np.sum((xv - xv.mean()) ** 2))
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
            fig.add_annotation(
                xref="x2 domain", yref="y2 domain", x=0.05, y=0.95,
                text=f"R\u00b2 = {r2:.3f}<br>n = {len(xv)}",
                showarrow=False, align="left",
                font=dict(size=13),
                bgcolor="rgba(255,255,255,0.75)",
                bordercolor="gray", borderwidth=1,
            )
 
    # --- Axis titles + layout ---
    fig.update_xaxes(title_text=x, row=1, col=1)
    fig.update_yaxes(title_text=y, row=1, col=1)
    fig.update_xaxes(title_text=f"{ref_cohort} {y}", row=1, col=2)
    fig.update_yaxes(title_text=f"{cmp_cohort} {y}", row=1, col=2)
    fig.update_layout(
        width=width, height=height,
        template="simple_white",
        legend=dict(title=cohort_col, x=0.0, y=1.0,
                    xanchor="left", yanchor="bottom", orientation="h"),
    )
    return fig

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_scan_pairs(
    df,
    x,
    y,
    group="scan_id",
    cohort_col="cohort",
    ref_cohort="milou",
    cmp_cohort="uc",
    colors=None,
    point_size=9,
    ky_size=4,
    line_color="rgba(150,150,150,0.55)",
    outliers=None,
    outlier_color="red",
    outlier_pad=6,
    width=1100,
    height=500,
):
    """
    Parameters
    ----------
    df : pandas.DataFrame
    x, y : str
        Column names for the left-panel axes. `y` is also the value compared
        in the parity panel.
    group : str
        Column whose shared values are joined by a line (default 'scan_id').
        Also the key used to pair cohorts.
    cohort_col : str
        Column holding the cohort label (default 'cohort').
    ref_cohort, cmp_cohort : str
        Cohort labels placed on the parity x- and y-axes respectively
        (default 'milou' on x, 'uc' on y). Matched case-insensitively. A
        milou/uc point appears on the left panel only if its scan_id has a
        value in both cohorts.
    colors : dict or None
        Overrides for point colors. Keys: ref_cohort, cmp_cohort, "KY", "other".
    point_size, ky_size : int
        Marker sizes for normal points and for KY-tagged points.
    line_color : str
        Color of the scan_id connecting lines.
    outliers : list of str or None
        scan_id values (matched against `group`, exact/case-sensitive) to flag.
        Each flagged point that is actually drawn gets a hollow square outline
        drawn around it in `outlier_color`, on both the left and parity panels.
    outlier_color : str
        Color of the outlier squares (default 'red').
    outlier_pad : int
        Pixels added to the underlying marker size to size the square, so the
        outline sits just outside the point it surrounds.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    colors = colors or {
        ref_cohort: "red",
        cmp_cohort: "blue",
        "KY": "green",
        "other": "lightgray",
    }
    outlier_ids = {str(s) for s in (outliers or [])}

    fig = make_subplots(rows=1, cols=2)

    # Cohort labels (case-insensitive) and the scan_ids paired across cohorts.
    coh = df[cohort_col].astype(str).str.strip().str.lower()
    ref = df[coh == ref_cohort.lower()].groupby(df[group])[y].mean()
    cmp = df[coh == cmp_cohort.lower()].groupby(df[group])[y].mean()
    pair = pd.concat([ref.rename("ref"), cmp.rename("cmp")], axis=1).dropna()
    paired_ids = set(pair.index)

    # --- Left: connecting lines for paired scan_ids only ---
    xs, ys = [], []
    for _, sub in df[df[group].isin(paired_ids)].groupby(group, sort=True):
        sub = sub.sort_values(x)
        xs.extend(sub[x].tolist() + [None])
        ys.extend(sub[y].tolist() + [None])
    fig.add_trace(
        go.Scatter(
            x=xs, y=ys, mode="lines",
            line=dict(color=line_color, width=1),
            hoverinfo="skip", showlegend=False,
        ),
        row=1, col=1,
    )

    # --- Left: points. milou/uc only when paired; KY overrides cohort color ---
    is_ky = df[group].astype(str).str.contains("KY", case=True, regex=False)
    cat = pd.Series("other", index=df.index)
    cat[coh == ref_cohort.lower()] = ref_cohort
    cat[coh == cmp_cohort.lower()] = cmp_cohort
    cat[is_ky] = "KY"

    style = {
        ref_cohort: (colors[ref_cohort], point_size, ref_cohort),
        cmp_cohort: (colors[cmp_cohort], point_size, cmp_cohort),
        "KY": (colors["KY"], ky_size, "KY (scan_id)"),
        "other": (colors["other"], point_size, "other"),
    }
    paired_mask = df[group].isin(paired_ids)
    for c, (color, size, name) in style.items():
        m = cat == c
        if c in (ref_cohort, cmp_cohort):      # red/blue must be paired
            m = m & paired_mask
        if not m.any():
            continue
        fig.add_trace(
            go.Scatter(
                x=df.loc[m, x], y=df.loc[m, y], mode="markers",
                marker=dict(color=color, size=size,
                            line=dict(width=0.5, color="white")),
                name=name,
                customdata=df.loc[m, [group, cohort_col]].to_numpy(),
                hovertemplate=(
                    f"{group}=%{{customdata[0]}}<br>{cohort_col}=%{{customdata[1]}}"
                    f"<br>{x}=%{{x}}<br>{y}=%{{y}}<extra></extra>"
                ),
            ),
            row=1, col=1,
        )

    # --- Left: red squares around outlier scan_ids (only points actually drawn) ---
    if outlier_ids:
        gstr = df[group].astype(str)
        drawn = (cat.isin([ref_cohort, cmp_cohort]) & paired_mask) | cat.isin(["KY", "other"])
        om = drawn & gstr.isin(outlier_ids)
        if om.any():
            # size each square just larger than the point it surrounds
            base = np.where(cat[om].to_numpy() == "KY", ky_size, point_size)
            fig.add_trace(
                go.Scatter(
                    x=df.loc[om, x], y=df.loc[om, y], mode="markers",
                    marker=dict(symbol="square-open", color=outlier_color,
                                size=base + outlier_pad,
                                line=dict(width=2, color=outlier_color)),
                    name="outlier",
                    customdata=df.loc[om, [group, cohort_col]].to_numpy(),
                    hovertemplate=(
                        f"outlier<br>{group}=%{{customdata[0]}}"
                        f"<br>{cohort_col}=%{{customdata[1]}}<extra></extra>"
                    ),
                ),
                row=1, col=1,
            )

    # --- Right: parity of y between cohorts, paired on `group` ---
    xv, yv = pair["ref"].to_numpy(), pair["cmp"].to_numpy()
    if len(xv):
        fig.add_trace(
            go.Scatter(
                x=xv, y=yv, mode="markers",
                marker=dict(color="black", size=7),
                text=pair.index.astype(str),
                hovertemplate=(
                    f"{group}=%{{text}}<br>{ref_cohort}=%{{x}}"
                    f"<br>{cmp_cohort}=%{{y}}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=1, col=2,
        )

        # red squares around outlier scan_ids present in the parity panel
        if outlier_ids:
            pom = [sid for sid in pair.index if str(sid) in outlier_ids]
            if pom:
                fig.add_trace(
                    go.Scatter(
                        x=pair.loc[pom, "ref"], y=pair.loc[pom, "cmp"],
                        mode="markers",
                        marker=dict(symbol="square-open", color=outlier_color,
                                    size=7 + outlier_pad,
                                    line=dict(width=2, color=outlier_color)),
                        text=[str(s) for s in pom],
                        hovertemplate=(
                            f"outlier<br>{group}=%{{text}}"
                            f"<br>{ref_cohort}=%{{x}}<br>{cmp_cohort}=%{{y}}<extra></extra>"
                        ),
                        showlegend=False,
                    ),
                    row=1, col=2,
                )

        lo = float(min(xv.min(), yv.min()))
        hi = float(max(xv.max(), yv.max()))
        pad = 0.05 * (hi - lo or 1.0)
        lo, hi = lo - pad, hi + pad

        fig.add_trace(
            go.Scatter(
                x=[lo, hi], y=[lo, hi], mode="lines",
                line=dict(color="gray", dash="dash", width=1),
                showlegend=False, hoverinfo="skip",
            ),
            row=1, col=2,
        )
        fig.update_xaxes(range=[lo, hi], row=1, col=2)
        fig.update_yaxes(range=[lo, hi], scaleanchor="x2", scaleratio=1,
                         row=1, col=2)

        if len(xv) >= 2:
            ss_res = float(np.sum((xv - yv) ** 2))
            ss_tot = float(np.sum((xv - xv.mean()) ** 2))
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
            fig.add_annotation(
                xref="x2 domain", yref="y2 domain", x=0.05, y=0.95,
                text=f"R\u00b2 = {r2:.3f}<br>n = {len(xv)}",
                showarrow=False, align="left",
                font=dict(size=13),
                bgcolor="rgba(255,255,255,0.75)",
                bordercolor="gray", borderwidth=1,
            )

    # --- Axis titles + layout ---
    fig.update_xaxes(title_text=x, row=1, col=1)
    fig.update_yaxes(title_text=y, row=1, col=1)
    fig.update_xaxes(title_text=f"{ref_cohort} {y}", row=1, col=2)
    fig.update_yaxes(title_text=f"{cmp_cohort} {y}", row=1, col=2)
    fig.update_layout(
        width=width, height=height,
        template="simple_white",
        legend=dict(title=cohort_col, x=0.0, y=1.0,
                    xanchor="left", yanchor="bottom", orientation="h"),
    )
    return fig

In [ ]:
milou = pd.read_csv(r"Z:\aorta\vasculaid\camp\camp2\milou_camp_sweep.csv")
uc = pd.read_csv(r"Z:\aorta\vasculaid\camp\camp2\UC_camp_sweep.csv")
milou['cohort'] = 'milou'
uc['cohort'] = 'uc'
data = pd.concat([milou, uc], ignore_index=True)
data = data[~data["scan_id"].isin(["KK40_1", "KK40_2", "KK40_3", "KK32_0"])].reset_index(drop=True)
data2 = data[data['n_patches_trimmed']==0]
plot_scan_pairs(data2, x="area_total", y="k_fluct",
                outliers=["KK16_6", "KK24_1", "KK14_1", "KK34_1", "KK38_5", "KK22_1"])

# Analysis

In [ ]:
import pandas as pd
xl = pd.ExcelFile(r'Z:\aorta\meta_data\meta.xlsx')
cohort_sheets = [s for s in xl.sheet_names if s != 'KEY']

results = []

for sheet in cohort_sheets:
    df = xl.parse(sheet)
    
    # Drop rows with no Patient_ID
    df = df[df['Patient_ID'].notna()]
    
    n_scans = len(df)
    unique_patients = df['Patient_ID'].nunique()
    avg_scans = n_scans / unique_patients if unique_patients > 0 else 0
    
    # Preop/postop: 1 = preop, 0 = postop
    if 'Preop' in df.columns:
        has_preop = set(df[df['Preop'] == 1]['Patient_ID'])
        has_postop = set(df[df['Preop'] == 0]['Patient_ID'])
        both = len(has_preop & has_postop)
    else:
        both = 'N/A'
    
    results.append({
        'Cohort': sheet,
        'Total Scans': n_scans,
        'Unique Patients': unique_patients,
        'Avg Scans/Patient': round(avg_scans, 2),
        'Patients w/ ≥1 Pre + ≥1 Post': both,
    })

summary = pd.DataFrame(results)
print(summary.to_string(index=False))

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------- scan-id parsing (from meta-data notebook) ----------
def scan_timepoint(scan_id):
    if pd.isna(scan_id): return np.nan
    m = re.search(r"_(\d+)$", str(scan_id).strip())
    return int(m.group(1)) if m else np.nan

def patient_from_scan(scan_id):
    if pd.isna(scan_id): return np.nan
    s = str(scan_id).strip(); m = re.match(r"(.+)_\d+$", s)
    return m.group(1) if m else s

def cohort_from_scan(scan_id):
    if pd.isna(scan_id): return "Unknown"
    m = re.match(r"^([A-Za-z]+)", str(scan_id).strip())
    return m.group(1).upper() if m else "Unknown"

# ---------- build one-row-per-scan table for a parameter combo ----------
def build_camp_table(df_sweep, r2, eta, mps, seed=None,
                     x_feature="surface_area", y_feature="k_fluct_dimless",
                     normalize_to_cohort=None):
    """Filter sweep master to one combo -> one row per scan.
    seed=None averages over all seeds (mean±std, like averaging over prefactor);
    seed=<value> pins one seed (std=0). Output mirrors the meta-data columns."""
    d = df_sweep.copy()
    d["scan_id"] = d["scan_id"].astype(str).str.strip()
    sel = (d["r2_thresh"] == r2) & (d["eta_thresh"] == eta) & (d["min_patch_size"] == mps)
    if seed is not None:
        sel &= (d["seed_patches"] == seed)
    d = d[sel].copy()
    if len(d) == 0:
        raise ValueError(f"No rows for r2={r2}, eta={eta}, mps={mps}, seed={seed}")
    for col, label in [(x_feature, "x_feature"), (y_feature, "y_feature")]:
        if col not in d.columns:
            raise KeyError(f"{label} '{col}' not in sweep columns. Available numeric:\n"
                           + ", ".join(c for c in d.columns if pd.api.types.is_numeric_dtype(d[c])))
    g = d.groupby("scan_id", dropna=True)
    out = g.agg(
        SurfaceArea_mean=(x_feature, "mean"), SurfaceArea_std=(x_feature, "std"),
        IntGaussian_Fluct_mean=(y_feature, "mean"), IntGaussian_Fluct_std=(y_feature, "std"),
        N_seeds=(x_feature, "size"),
    ).reset_index().rename(columns={"scan_id": "Scan_ID"})
    out["SurfaceArea_std"] = out["SurfaceArea_std"].fillna(0.0)
    out["IntGaussian_Fluct_std"] = out["IntGaussian_Fluct_std"].fillna(0.0)
    out["Cohort"] = out["Scan_ID"].map(cohort_from_scan)
    out["Timepoint"] = out["Scan_ID"].map(scan_timepoint)
    out["Patient_Key"] = out["Scan_ID"].map(patient_from_scan)
    first_tp = out.groupby("Patient_Key")["Timepoint"].transform("min")
    out["Is_Preop"] = out["Timepoint"].eq(first_tp)   # preop = earliest scan per patient
    if normalize_to_cohort is not None:
        ref = out[out["Cohort"] == normalize_to_cohort]
        ra, rk = ref["SurfaceArea_mean"].mean(), ref["IntGaussian_Fluct_mean"].mean()
        if ra and rk:
            for c in ["SurfaceArea_mean", "SurfaceArea_std"]: out[c] /= ra
            for c in ["IntGaussian_Fluct_mean", "IntGaussian_Fluct_std"]: out[c] /= rk
    return out

def attach_meta(table, meta_df):
    m = meta_df.copy()
    m["Scan_ID"] = m["Scan_ID"].astype(str).str.strip()
    m["Days_From_Proc"] = pd.to_numeric(m.get("Days_From_Proc"), errors="coerce")
    keep = [c for c in ["Scan_ID", "Patient_ID", "Days_From_Proc"] if c in m.columns]
    m = m[keep].sort_values("Scan_ID").drop_duplicates("Scan_ID", keep="first")
    return table.merge(m, on="Scan_ID", how="left")

# ---------- shared plot primitives ----------
COHORT_PALETTE = [
    "#E15759", "#4E79A7", "#59A14F", "#B07AA1", "#F28E2B", "#76B7B2",
    "#EDC948", "#9C755F", "#FF9DA7", "#BAB0AC", "#8CD17D", "#499894",
    "#86BCB6", "#D37295", "#FABFD2", "#B6992D",
]
X_AXIS_LABEL, Y_AXIS_LABEL = "Area", "δK"

def _cohorts(t): return sorted(t["Cohort"].dropna().astype(str).unique().tolist())

def _bg_trace(df, visible=True):
    return go.Scatter(
        x=df["SurfaceArea_mean"], y=df["IntGaussian_Fluct_mean"],
        error_x=dict(type="data", array=df["SurfaceArea_std"], thickness=1, color="rgba(120,120,120,0.55)"),
        error_y=dict(type="data", array=df["IntGaussian_Fluct_std"], thickness=1, color="rgba(120,120,120,0.55)"),
        mode="markers", marker=dict(size=7, color="rgba(130,130,130,0.70)"),
        showlegend=False, visible=visible,
        customdata=np.column_stack([df["Scan_ID"], df["Cohort"]]),
        hovertemplate="Scan: %{customdata[0]}<br>Cohort: %{customdata[1]}<br>Area: %{x:.3f}<br>δK: %{y:.3f}<extra></extra>")

def _highlight_trace(df, color="black", visible=True):
    return go.Scatter(
        x=df["SurfaceArea_mean"], y=df["IntGaussian_Fluct_mean"],
        error_x=dict(type="data", array=df["SurfaceArea_std"], thickness=1.3, color=color),
        error_y=dict(type="data", array=df["IntGaussian_Fluct_std"], thickness=1.3, color=color),
        mode="markers", marker=dict(size=10, color="white", line=dict(color=color, width=2.1)),
        showlegend=False, visible=visible,
        customdata=np.column_stack([df["Scan_ID"], df["Cohort"], df["N_seeds"]]),
        hovertemplate="Scan: %{customdata[0]}<br>Cohort: %{customdata[1]}<br>Area: %{x:.3f}<br>δK: %{y:.3f}<br>N seeds: %{customdata[2]}<extra></extra>")

def _path_traces(df, connect_all_points, visible=True):
    traces = []
    for patient_key, grp in df.groupby("Patient_Key"):
        grp = grp.sort_values("Timepoint").copy()
        if len(grp) == 0: continue
        if connect_all_points:
            path = grp
        else:
            pre, post = grp[grp["Is_Preop"]], grp[~grp["Is_Preop"]]
            if len(pre) == 0: continue
            path = pre.iloc[[0]].copy() if len(post) == 0 else pd.concat([pre.iloc[[0]], post.iloc[[-1]]], ignore_index=True)
        edge_colors = ["black" if p else "red" for p in path["Is_Preop"]]
        traces.append(go.Scatter(
            x=path["SurfaceArea_mean"], y=path["IntGaussian_Fluct_mean"],
            mode="lines+markers" if len(path) > 1 else "markers",
            marker=dict(size=10, color="white", line=dict(color=edge_colors, width=2.6)),
            line=dict(color="black", width=2.0), showlegend=False, visible=visible,
            customdata=np.column_stack([path["Scan_ID"], path["Cohort"]]),
            hovertemplate="Patient: " + str(patient_key) + "<br>Scan: %{customdata[0]}<br>Cohort: %{customdata[1]}<br>Area: %{x:.3f}<br>δK: %{y:.3f}<extra></extra>"))
    return traces

# ---------- static scatter ----------
def build_camp_scatter(table, mode):
    """mode in {preop_only, preop_postop, trajectories_all, trajectories_first_last}."""
    fig = go.Figure(); cohorts = _cohorts(table)
    state_traces = {s: [] for s in ["All"] + cohorts}
    def add_mode_content(df, visible):
        added = []
        if mode == "preop_only":
            fig.add_trace(_highlight_trace(df[df["Is_Preop"]], "black", visible)); added.append(len(fig.data) - 1)
        elif mode == "preop_postop":
            fig.add_trace(_highlight_trace(df[df["Is_Preop"]], "black", visible)); added.append(len(fig.data) - 1)
            post = df[~df["Is_Preop"]]
            if len(post): fig.add_trace(_highlight_trace(post, "red", visible)); added.append(len(fig.data) - 1)
        elif mode == "trajectories_all":
            for t in _path_traces(df.dropna(subset=["Timepoint"]), True, visible): fig.add_trace(t); added.append(len(fig.data) - 1)
        elif mode == "trajectories_first_last":
            for t in _path_traces(df.dropna(subset=["Timepoint"]), False, visible): fig.add_trace(t); added.append(len(fig.data) - 1)
        return added
    fig.add_trace(_bg_trace(table, True)); state_traces["All"].append(len(fig.data) - 1)
    state_traces["All"] += add_mode_content(table, True)
    for cohort in cohorts:
        fig.add_trace(_bg_trace(table, False)); state_traces[cohort].append(len(fig.data) - 1)
        state_traces[cohort] += add_mode_content(table[table["Cohort"].astype(str) == cohort], False)
    buttons = []
    for state in ["All"] + cohorts:
        vis = [False] * len(fig.data)
        for k in state_traces[state]: vis[k] = True
        buttons.append(dict(label=state, method="update", args=[{"visible": vis}, {"title": f"CAMP | {mode} | Highlight: {state}"}]))
    fig.update_layout(template="plotly_white", width=760, height=620, font=dict(size=14, color="black"),
                      margin=dict(l=70, r=30, t=95, b=65), title=f"CAMP | {mode} | Highlight: All",
                      xaxis_title=X_AXIS_LABEL, yaxis_title=Y_AXIS_LABEL,
                      updatemenus=[dict(buttons=buttons, direction="down", x=0.5, y=1.14, xanchor="center", yanchor="top")])
    return fig

# ---------- derivative table + scatter ----------
def interval_bin(m):
    if pd.isna(m): return np.nan
    if m < 3: return "0-3 months"
    if m < 9: return "3-9 months"
    if m < 24: return "9-24 months"
    return "24+ months"

def build_derivative_table(table_with_time):
    rows = []; df = table_with_time.dropna(subset=["Days_From_Proc"]).copy()
    for patient_key, grp in df.groupby("Patient_Key"):
        grp = grp.sort_values("Days_From_Proc")
        if len(grp) < 2: continue
        for i in range(len(grp) - 1):
            a, b = grp.iloc[i], grp.iloc[i + 1]
            d_days = b["Days_From_Proc"] - a["Days_From_Proc"]
            if pd.isna(d_days) or d_days <= 0: continue
            d_months = d_days / 30.4375; end_month = max(b["Days_From_Proc"], 0) / 30.4375
            rows.append(dict(Cohort=a["Cohort"], Patient_Key=patient_key, Start_Scan=a["Scan_ID"], End_Scan=b["Scan_ID"],
                Delta_Months=d_months, End_Months_From_Proc=end_month,
                dArea_dt=(b["SurfaceArea_mean"] - a["SurfaceArea_mean"]) / d_months,
                dK_dt=(b["IntGaussian_Fluct_mean"] - a["IntGaussian_Fluct_mean"]) / d_months,
                Interval_Bin=interval_bin(end_month)))
    return pd.DataFrame(rows)

def build_camp_derivative(deriv, bucketed=False):
    if len(deriv) == 0:
        raise ValueError("No derivative intervals (need >=2 timepoints per patient with Days_From_Proc).")
    cohorts = sorted(deriv["Cohort"].dropna().astype(str).unique().tolist())
    cohort_colors = {c: COHORT_PALETTE[i % len(COHORT_PALETTE)] for i, c in enumerate(cohorts)}
    max_x = float(deriv["dArea_dt"].abs().max()) * 1.15 or 1.0
    max_y = float(deriv["dK_dt"].abs().max()) * 1.15 or 1.0
    fig = go.Figure()
    null_x, null_y = float(deriv["dArea_dt"].abs().mean() / 3), float(deriv["dK_dt"].abs().mean() / 3)
    fig.add_shape(type="rect", x0=-null_x, x1=null_x, y0=-null_y, y1=null_y, line=dict(width=0), fillcolor="rgba(160,160,160,0.30)", layer="below")
    fig.add_shape(type="line", x0=0, x1=0, y0=-max_y, y1=max_y, line=dict(color="black", width=1.2), layer="below")
    fig.add_shape(type="line", x0=-max_x, x1=max_x, y0=0, y1=0, line=dict(color="black", width=1.2), layer="below")
    intervals = ["All intervals", "0-3 months", "3-9 months", "9-24 months", "24+ months"] if bucketed else ["All intervals"]
    def mask_interval(d, name):
        return pd.Series(True, index=d.index) if name == "All intervals" else d["Interval_Bin"].astype(str) == name
    state_traces = {}
    for state in ["All"] + cohorts:
        for iv in intervals:
            key = (state, iv); state_traces[key] = []; d = deriv[mask_interval(deriv, iv)]
            if state == "All":
                for c in cohorts:
                    sub = d[d["Cohort"].astype(str) == c]
                    if len(sub) == 0: continue
                    fig.add_trace(go.Scatter(x=sub["dArea_dt"], y=sub["dK_dt"], mode="markers", name=c,
                        legendgroup=c, showlegend=(iv == intervals[0]),
                        marker=dict(size=9, color=cohort_colors[c], line=dict(color="black", width=0.8)), visible=False,
                        customdata=np.column_stack([sub["Patient_Key"], sub["Start_Scan"], sub["End_Scan"], sub["Delta_Months"].round(2)]),
                        hovertemplate="Patient: %{customdata[0]}<br>%{customdata[1]} → %{customdata[2]}<br>Δmonths: %{customdata[3]}<br>dArea/dt: %{x:.4f}<br>dδK/dt: %{y:.4f}<extra></extra>"))
                    state_traces[key].append(len(fig.data) - 1)
            else:
                fig.add_trace(go.Scatter(x=d["dArea_dt"], y=d["dK_dt"], mode="markers",
                    marker=dict(size=7, color="rgba(145,145,145,0.62)"), showlegend=False, visible=False,
                    customdata=np.column_stack([d["Cohort"], d["Patient_Key"]]),
                    hovertemplate="Cohort: %{customdata[0]}<br>Patient: %{customdata[1]}<br>dArea/dt: %{x:.4f}<br>dδK/dt: %{y:.4f}<extra></extra>"))
                state_traces[key].append(len(fig.data) - 1)
                hi = d[d["Cohort"].astype(str) == state]
                if len(hi):
                    fig.add_trace(go.Scatter(x=hi["dArea_dt"], y=hi["dK_dt"], mode="markers", name=state,
                        legendgroup=state, showlegend=(iv == intervals[0]),
                        marker=dict(size=10, color=cohort_colors[state], line=dict(color="black", width=1.1)), visible=False,
                        customdata=np.column_stack([hi["Patient_Key"], hi["Start_Scan"], hi["End_Scan"]]),
                        hovertemplate="Patient: %{customdata[0]}<br>%{customdata[1]} → %{customdata[2]}<br>dArea/dt: %{x:.4f}<br>dδK/dt: %{y:.4f}<extra></extra>"))
                    state_traces[key].append(len(fig.data) - 1)
    for k in state_traces[("All", intervals[0])]: fig.data[k].visible = True
    buttons = []
    for state in ["All"] + cohorts:
        for iv in intervals:
            vis = [False] * len(fig.data)
            for k in state_traces[(state, iv)]: vis[k] = True
            label = state if not bucketed else f"{state} | {iv}"
            buttons.append(dict(label=label, method="update", args=[{"visible": vis}, {"title": f"CAMP derivative | {label}"}]))
    fig.update_layout(template="plotly_white", width=780, height=650, font=dict(size=14, color="black"),
                      margin=dict(l=70, r=40, t=95, b=70),
                      title=f"CAMP derivative | {'All | All intervals' if bucketed else 'All'}",
                      xaxis=dict(title="dArea / month", range=[-max_x, max_x]), yaxis=dict(title="dδK / month", range=[-max_y, max_y]),
                      updatemenus=[dict(buttons=buttons, direction="down", x=0.5, y=1.14, xanchor="center", yanchor="top")])
    return fig

# ---------- temporal (A & δK stacked) ----------
def build_camp_temporal(table_with_time, use_months=True):
    df = table_with_time.dropna(subset=["Days_From_Proc"]).copy()
    if len(df) == 0:
        raise ValueError("No scans with Days_From_Proc for the temporal plot.")
    df["TimeAxis"] = df["Days_From_Proc"] / 30.4375 if use_months else df["Days_From_Proc"]
    x_label = "Time From Procedure (months)" if use_months else "Time From Procedure (days)"
    cohorts = _cohorts(df); cohort_colors = {c: COHORT_PALETTE[i % len(COHORT_PALETTE)] for i, c in enumerate(cohorts)}
    rows = [("SurfaceArea_mean", "SurfaceArea_std", "A"), ("IntGaussian_Fluct_mean", "IntGaussian_Fluct_std", "δK")]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09, subplot_titles=("A vs Time", "δK vs Time"))
    state_traces = {s: [] for s in ["All"] + cohorts}
    for _, grp in df.groupby("Patient_Key"):
        grp = grp.sort_values("Days_From_Proc")
        if len(grp) < 2: continue
        for r, (yc, _, _) in enumerate(rows, start=1):
            fig.add_trace(go.Scatter(x=grp["TimeAxis"], y=grp[yc], mode="lines",
                line=dict(color="rgba(0,0,0,0.20)", width=1), showlegend=False, hoverinfo="skip"), row=r, col=1)
    def add_points(sub, color, r, yc, yerr, ylabel, visible, name, show_legend):
        fig.add_trace(go.Scatter(x=sub["TimeAxis"], y=sub[yc],
            error_y=dict(type="data", array=sub[yerr].fillna(0), thickness=1.2, color=color),
            mode="markers", name=name, legendgroup=name, showlegend=show_legend, visible=visible,
            marker=dict(size=8, color=color, line=dict(color="black", width=0.7)),
            customdata=np.column_stack([sub["Scan_ID"], sub["Patient_Key"]]),
            hovertemplate="Scan: %{customdata[0]}<br>Patient: %{customdata[1]}<br>Time: %{x:.2f}<br>" + ylabel + ": %{y:.3f}<extra></extra>"), row=r, col=1)
        return len(fig.data) - 1
    for c in cohorts:
        sub = df[df["Cohort"] == c]
        if len(sub) == 0: continue
        for r, (yc, yerr, ylabel) in enumerate(rows, start=1):
            state_traces["All"].append(add_points(sub, cohort_colors[c], r, yc, yerr, ylabel, True, c, (r == 1)))
    for c in cohorts:
        for r, (yc, yerr, ylabel) in enumerate(rows, start=1):
            state_traces[c].append(add_points(df, "rgba(145,145,145,0.55)", r, yc, yerr, ylabel, False, f"bg_{c}", False))
            hi = df[df["Cohort"] == c]
            if len(hi): state_traces[c].append(add_points(hi, cohort_colors[c], r, yc, yerr, ylabel, False, c, (r == 1)))
    for r in (1, 2): fig.add_vline(x=0, line_dash="dash", line_color="black", line_width=1.2, row=r, col=1)
    fig.update_xaxes(title_text=x_label, row=2, col=1)
    fig.update_yaxes(title_text="A", row=1, col=1); fig.update_yaxes(title_text="δK", row=2, col=1)
    buttons = []
    for state in ["All"] + cohorts:
        vis = [t.mode == "lines" for t in fig.data]   # keep faint trajectory lines on
        for k in state_traces[state]: vis[k] = True
        buttons.append(dict(label=state, method="update", args=[{"visible": vis}, {"title": f"CAMP temporal A & δK | Highlight: {state}"}]))
    fig.update_layout(template="plotly_white", width=820, height=820, font=dict(size=13, color="black"),
                      margin=dict(l=80, r=30, t=100, b=70), title="CAMP temporal A & δK | Highlight: All",
                      legend=dict(font=dict(size=12), bgcolor="rgba(255,255,255,0.85)"),
                      updatemenus=[dict(buttons=buttons, direction="down", x=0.5, y=1.08, xanchor="center", yanchor="top")])
    return fig

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

UC_SWEEP_CSV = r"Z:\aorta\thoracic\camp_sweeps\UC_camp_sweep.csv"
STAR_SCAN_ID = "MF1_1"

uc = pd.read_csv(UC_SWEEP_CSV)
uc["scan_id"] = uc["scan_id"].astype(str).str.strip()
uc = uc[uc["scan_id"].str.startswith(("KK", "KY"), na=False)].copy()

area_candidates = [
    "area_total",
    "surface_area_mm2",
    "surface_area",
    "area",
    "mesh_area",
]
area_col = next((c for c in area_candidates if c in uc.columns), None)
if area_col is None:
    raise KeyError(f"Could not find an area column in UC_camp_sweep.csv. Tried: {area_candidates}")

if "k_std_dimless" not in uc.columns:
    raise KeyError("Could not find 'k_std_dimless' in UC_camp_sweep.csv.")

def _pick_star_source():
    if "csweep" in globals() and isinstance(csweep, pd.DataFrame) and not csweep.empty:
        return csweep
    if "camp_tbl" in globals() and isinstance(camp_tbl, pd.DataFrame) and not camp_tbl.empty:
        return camp_tbl
    return None

star_source = _pick_star_source()
if star_source is None:
    raise ValueError("Need the MRA_French sweep dataframe loaded in the notebook so MF1_1 can be starred.")

star_source = star_source.copy()
star_source["scan_id"] = star_source["scan_id"].astype(str).str.strip()
star_row = star_source.loc[star_source["scan_id"] == STAR_SCAN_ID]
if star_row.empty:
    raise ValueError(f"{STAR_SCAN_ID} was not found in the available star source dataframe.")
star_row = star_row.iloc[0]

star_area_col = next((c for c in area_candidates if c in star_source.columns and c in uc.columns), None)
if star_area_col is None:
    raise KeyError(
        "Could not find a common area column between the UC sweep and the MRA_French sweep. "
        f"UC columns: {sorted(uc.columns.tolist())}"
    )

star_x = float(star_row[star_area_col])
star_y = float(star_row["k_std_dimless"])

fig = go.Figure()

ky_mask = uc["scan_id"].str.startswith("KY", na=False)
kk_mask = uc["scan_id"].str.startswith("KK", na=False)

fig.add_trace(
    go.Scatter(
        x=uc.loc[kk_mask, area_col],
        y=uc.loc[kk_mask, "k_std_dimless"],
        mode="markers",
        name="KK",
        marker=dict(color="red", size=8, opacity=0.7),
        text=uc.loc[kk_mask, "scan_id"],
        hovertemplate="%{text}<br>area=%{x:.4g}<br>k_std_dimless=%{y:.4g}<extra>KK</extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=uc.loc[ky_mask, area_col],
        y=uc.loc[ky_mask, "k_std_dimless"],
        mode="markers",
        name="KY",
        marker=dict(color="green", size=8, opacity=0.7),
        text=uc.loc[ky_mask, "scan_id"],
        hovertemplate="%{text}<br>area=%{x:.4g}<br>k_std_dimless=%{y:.4g}<extra>KY</extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=[star_x],
        y=[star_y],
        mode="markers+text",
        name=STAR_SCAN_ID,
        text=[STAR_SCAN_ID],
        textposition="top center",
        marker=dict(symbol="star", color="black", size=18, line=dict(color="black", width=1)),
        hovertemplate=f"{STAR_SCAN_ID}<br>area=%{{x:.4g}}<br>k_std_dimless=%{{y:.4g}}<extra>MF1_1</extra>",
    )
)

fig.update_layout(
    title="Area vs k_std_dimless: UC sweeps with MF1_1 reference",
    xaxis_title=area_col,
    yaxis_title="k_std_dimless",
    template="plotly_white",
    width=900,
    height=650,
    legend=dict(title="Scan group"),
)

fig.show()

In [ ]:
# ============ SET YOUR SWEEP COMBO HERE ============
SEL_R2, SEL_ETA, SEL_MPS = 0.9, 0.075, 18
SEL_SEED = 100
X_FEATURE = "surface_area_mm2"
Y_FEATURE = "k_fluct"
NORMALIZE_TO_COHORT = "KY"
SWEEP_CSV = r"Z:\aorta\thoracic\camp_sweeps2\sweep_master.csv"
META_XLSX = r"Z:\aorta\meta_data\meta.xlsx"
# ===================================================

try:
    df = csweep.copy()
except NameError:
    df = pd.read_csv(SWEEP_CSV)
df["scan_id"] = df["scan_id"].astype(str).str.strip()
df = df[df['euler_number'] == 0]  # filter out non-manifold meshes

PARAM_COLS = ["r2_thresh", "eta_thresh", "min_patch_size", "seed_patches"]
def _pick(cands): return next((c for c in cands if c in df.columns), None)
X_FEATURE = X_FEATURE or _pick(["surface_area_mm2", "surface_area", "area", "total_area", "mesh_area", "SurfaceArea"])
Y_FEATURE = Y_FEATURE or _pick(["k_fluct", "IntGaussian_Fluct", "k_std_dimless", "k_fluct_dimless", "k_fluct_aw"])
print("X feature:", X_FEATURE, "| Y feature:", Y_FEATURE)
print("available numeric features:",
      [c for c in df.columns if c not in PARAM_COLS + ["scan_id"] and pd.api.types.is_numeric_dtype(df[c])])

camp_tbl = build_camp_table(df, SEL_R2, SEL_ETA, SEL_MPS, seed=SEL_SEED,
                            x_feature=X_FEATURE, y_feature=Y_FEATURE,
                            normalize_to_cohort=NORMALIZE_TO_COHORT)
print(f"\nCAMP table: {len(camp_tbl)} scans | combo r2={SEL_R2} eta={SEL_ETA} mps={SEL_MPS} seed={SEL_SEED}")

summary = (camp_tbl.groupby("Cohort")
           .agg(Total_Scans=("Scan_ID", "nunique"),
                Unique_Patients=("Patient_Key", "nunique"),
                Preop_Scans=("Is_Preop", "sum")).reset_index())
summary["Avg_Scans_per_Patient"] = (summary.Total_Scans / summary.Unique_Patients).round(2)
print(summary.to_string(index=False))

try:
    mx = pd.ExcelFile(META_XLSX)
    sheets = [s for s in mx.sheet_names if s != "KEY"]
    meta = pd.concat([mx.parse(s) for s in sheets], ignore_index=True)
    if "Scan_ID" not in meta.columns:
        sid = [c for c in meta.columns if c.strip().lower().replace(" ", "_") == "scan_id"]
        if sid: meta = meta.rename(columns={sid[0]: "Scan_ID"})
    camp_tbl = attach_meta(camp_tbl, meta)
except Exception as e:
    print(f"Meta merge skipped: {e}")

print("\nWith meta (head):")
display(camp_tbl.head())

camp_tbl.to_csv(r"Z:\aorta\thoracic\camp_table_seedavg.csv", index=False)
print("Saved -> Z:\\aorta\\thoracic\\camp_table_seedavg.csv")

In [ ]:
for mode in ["preop_only", "preop_postop", "trajectories_all", "trajectories_first_last"]:
    build_camp_scatter(camp_tbl, mode).show()

In [ ]:
deriv = build_derivative_table(camp_tbl_t)
print("derivative intervals:", len(deriv))
if len(deriv):
    build_camp_derivative(deriv, bucketed=False).show()
    build_camp_derivative(deriv, bucketed=True).show()

In [ ]:
build_camp_temporal(camp_tbl_t).show()